# 02a - Data collection

**Laboratory of Bioinformatics 2 - Group 10**

The aim of this practical session is to collect a preliminary dataset for training and testing methods that predict the presence of a signal peptide (SP). We consider only eukaryotic proteins: the positive set contains proteins with an experimentally determined SP, and the negative set contains proteins without an annotated SP.

Following **Practical Session I - Part A** (slide 32), we retrieve entries from UniProtKB, apply the selection criteria, and save each dataset as a TSV file and a FASTA file. Redundancy reduction and splitting for cross-validation and independent testing are later preparation steps; they are not performed here.

**Course material:** `02a-DataCollection-2026.pdf` (slides 18-19 and 25-32), `02a-1-PythonGenerators.ipynb`, and `02a-3-UniProtAPI-examples.ipynb`.

Run the cells in order in a Python environment with `requests` installed and an internet connection. Open this notebook from a local checkout of the repository. See [the project README](../README.md) for the folder structure and setup.

## 1. Configuration

We use the UniProt REST API to retrieve entries programmatically. An HTTP GET request sends a search to the server, which returns a structured response in JSON format. This lets us inspect annotations and extract the fields needed for our dataset, as in the API examples discussed in class.

### 1.1 Import the required modules

We use `requests` to make HTTP requests, `json` to read the response body, and `re` to extract the next-page link. `HTTPAdapter` and `Retry` are used to handle temporary connection or service failures, following the UniProt API example notebook.

`Path` from Python's standard library is used to locate the project folders and save outputs in `data/collected/`.

In [ ]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re
from pathlib import Path

### 1.2 Configure the connection

The course example uses a `Session` with a maximum of five retries for the listed temporary server errors. We use the same configuration.

The `uniprotkb/search` endpoint returns entries in batches. This allows us to retrieve a large result set one page at a time, rather than loading the entire search in a single response (slides 9-10 and 14).

In [2]:
# Retry temporary server errors, as in the course example.
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

uniprot_url = "https://rest.uniprot.org/uniprotkb/search"

### 1.3 Define the search queries

The common criteria from slide 32 are included in both queries:

| Criterion | Query field |
| --- | --- |
| Reviewed entries | `reviewed:true` |
| No fragments | `fragment:false` |
| Eukaryotic proteins | `taxonomy_id:2759` |
| Evidence at protein level | `existence:1` |
| Protein length of at least 40 residues | `length:[40 TO *]` |

For **positive data**, we also require experimental signal-peptide evidence (`ft_signal_exp:*`). The cleavage-site and SP-length checks are applied to the returned JSON entries.

For **negative data**, we exclude any signal-peptide annotation (`NOT (ft_signal:*)`) and require experimental subcellular localization in one of the compartments specified in class:

| Compartment | UniProt location ID |
| --- | --- |
| Cytosol | `SL-0091` |
| Mitochondrion | `SL-0173` |
| Nucleus | `SL-0191` |
| Plastid | `SL-0209` |
| Peroxisome | `SL-0204` |
| Cell membrane | `SL-0039` |

The `cc_scl_term_exp` field requests experimental localization evidence. As explained on slide 27, membrane proteins without an SP belong in the negative set too.

We keep the queries readable and use `requests` to encode them into search URLs. The response format is JSON and the batch size is 500. This replaces pasting a URL generated by the UniProt website; the batch retrieval follows the same approach as the course notebook.

In [3]:
positive_query = """
(reviewed:true) AND
(fragment:false) AND
(taxonomy_id:2759) AND
(existence:1) AND
(length:[40 TO *]) AND
(ft_signal_exp:*)
""".replace("\n", " ")


negative_query = """
(reviewed:true) AND
(fragment:false) AND
(taxonomy_id:2759) AND
(existence:1) AND
(length:[40 TO *]) NOT
(ft_signal:*) AND
(
    cc_scl_term_exp:SL-0091 OR
    cc_scl_term_exp:SL-0173 OR
    cc_scl_term_exp:SL-0191 OR
    cc_scl_term_exp:SL-0209 OR
    cc_scl_term_exp:SL-0204 OR
    cc_scl_term_exp:SL-0039
)
""".replace("\n", " ")

# Build the search URLs from the queries, keeping the criteria readable above.
batch_size = 500
positive_url = requests.Request("GET", uniprot_url, params={
    "query": positive_query,
    "format": "json",
    "compressed": "false",
    "size": batch_size,
}).prepare().url

negative_url = requests.Request("GET", uniprot_url, params={
    "query": negative_query,
    "format": "json",
    "compressed": "false",
    "size": batch_size,
}).prepare().url

## 2. Retrieve entries, apply filters, and extract fields

We follow the structure of `02a-3-UniProtAPI-examples.ipynb`: a generator retrieves the batches, a filter function decides whether to keep an entry, and an extraction function returns the fields to write.

As in the coiled-coil example, criteria that need inspection of individual features are checked after the API response is received. Here we inspect signal-peptide and transmembrane annotations instead of coiled-coil regions.

### 2.1 Handle pagination with a generator

`get_next_link` extracts the next-page URL from the HTTP `Link` header. The URL already contains the cursor needed to continue the search.

`get_batch` is a generator: `yield` returns one response and the total result count from `x-total-results`, then pauses until the loop requests another batch. It continues until there is no next-page link. This is the same generator and pagination pattern used in the class notebooks (slides 14-15).

`raise_for_status()` reports unsuccessful HTTP requests. This notebook also sets a 60-second timeout for each request.

In [4]:
def get_next_link(headers):
    if "Link" in headers:
        # Extract the next-page URL from the Link header.
        match = re.search(r'<([^>]+)>;\s*rel="next"', headers["Link"])
        if match:
            return match.group(1)
    return None


def get_batch(batch_url):
    while batch_url:
        # Run the API call and check for HTTP errors.
        response = session.get(batch_url, timeout=60)
        response.raise_for_status()

        total = response.headers["x-total-results"]
        yield response, total

        # Stop when there is no next-page link.
        batch_url = get_next_link(response.headers)

### 2.2 Extract the fields shared by both datasets

Slides 30-31 require the accession, organism name, eukaryotic kingdom, and protein length in both TSV files. We read these values from the JSON entry and return them as a tuple, as in the course example's `extract_fields` function.

The kingdom is obtained from the organism's taxonomic lineage. The code uses `Metazoa`, `Fungi`, `Viridiplantae`, and `Other`. **Naming note:** the slides call the plant category `Plants`; this notebook currently retains the UniProt lineage label `Viridiplantae`.

In [5]:
def get_kingdom(entry):
    lineage = entry.get("organism", {}).get("lineage", [])
    for kingdom in ("Metazoa", "Fungi", "Viridiplantae"):
        if kingdom in lineage:
            return kingdom
    return "Other"


def extract_common_fields(entry):
    # Return the fields shared by the positive and negative TSV files.
    accession = entry["primaryAccession"]
    organism = entry["organism"]["scientificName"]
    kingdom = get_kingdom(entry)
    length = entry["sequence"]["length"]
    return accession, organism, kingdom, length

### 2.3 Select a valid signal peptide

Signal peptides are stored in the JSON `features` list with type `Signal`. As described on slides 19 and 25-26, their annotations include the cleavage position and supporting evidence.

For the positive set, we look for the first signal feature that meets the practical-session criteria:

- Experimental evidence code `ECO:0000269`, supporting the annotation with published experiments.
- A cleavage position that is present and is not marked `UNKNOWN`.
- No `Not cleaved` description.
- An SP length of at least 14 residues.

The slides describe the signal peptide as starting at position 1, so its end position gives its length and the reported cleavage position. The code therefore checks that the end is at least 14. `get_valid_signal` returns the accepted feature, or `None` if no feature passes.

In [6]:
def get_valid_signal(entry):
    # Find the first signal peptide that meets all selection criteria.
    for feature in entry.get("features", []):
        if feature.get("type") != "Signal":
            continue
        if "Not cleaved" in feature.get("description", ""):
            continue

        end_info = feature.get("location", {}).get("end", {})
        end = end_info.get("value")
        if end is None or end_info.get("modifier") == "UNKNOWN":
            continue
        if end < 14:
            continue

        # Require experimental evidence for this specific feature.
        for evidence in feature.get("evidences", []):
            if evidence.get("evidenceCode") == "ECO:0000269":
                return feature
    return None

### 2.4 Identify an N-terminal transmembrane region

Slide 31 asks us to record whether a transmembrane helix starts within the first 90 residues. We inspect `Transmembrane` features and return `True` if a known start position is at most 90; otherwise we return `False`.

We do not remove these proteins from the negative set. As discussed in class, N-terminal TM regions have physicochemical similarities to signal peptides and can be difficult negative examples for the predictors.

In [7]:
def has_tm_in_first_90(entry):
    # Keep TM proteins in the negative set and record this flag.
    for feature in entry.get("features", []):
        if feature.get("type") == "Transmembrane":
            start = feature.get("location", {}).get("start", {}).get("value")
            if start is not None and start <= 90:
                return True
    return False

### 2.5 Separate filtering from field extraction

Following the course notebook, each filter returns `True` or `False`. The corresponding extraction function is called only when the entry passes the filter and returns a tuple for the TSV row.

| Dataset | Selection after retrieval | Dataset-specific field |
| --- | --- | --- |
| Positive | Keep an entry if `get_valid_signal` finds an accepted SP. | `cleavage_pos`: the end position of that same accepted SP. |
| Negative | Reject any entry with a `Signal` feature, regardless of its evidence code. | `has_tm_in_90`: the N-terminal TM flag. |

The shared criteria and experimental localization of negatives are already specified in the search queries. These functions apply the additional feature checks to the returned entries.

In [8]:
def filter_positive_entry(entry):
    return get_valid_signal(entry) is not None


def filter_negative_entry(entry):
    # Exclude signal peptides at any evidence level.
    for feature in entry.get("features", []):
        if feature.get("type") == "Signal":
            return False
    return True


def extract_positive_fields(entry):
    accession, organism, kingdom, length = extract_common_fields(entry)
    signal = get_valid_signal(entry)
    cleavage_pos = signal["location"]["end"]["value"]
    return accession, organism, kingdom, length, cleavage_pos


def extract_negative_fields(entry):
    accession, organism, kingdom, length = extract_common_fields(entry)
    has_tm_in_90 = has_tm_in_first_90(entry)
    return accession, organism, kingdom, length, has_tm_in_90

### 2.6 Collect and save the dataset

`get_dataset` follows the batch loop shown in the UniProt API example. It receives a search URL, a filter function, and an extraction function. Here we also pass column names and an output prefix because the practical session requires a TSV with a header and a matching FASTA file.

For each batch, `json.loads(batch.text)` reads the JSON response. We iterate over `results`, count the retrieved entries, apply the filter, and write the accepted proteins:

- **TSV:** the extracted fields are separated by tabs using `print(*fields, sep="\t")`.
- **FASTA:** the accession is written after `>`, followed by the full protein sequence.

Unlike the example's `filtered_json` list, this version writes accepted entries directly to the files. The same accepted entry supplies both its metadata and sequence. The final counts show how many entries were retrieved and how many passed the filter.

In [9]:
def get_dataset(search_url, filter_function, extract_function, columns, output_prefix):
    n_total, n_filtered = 0, 0
    tsv_path = f"{output_prefix}.tsv"
    fasta_path = f"{output_prefix}.fasta"

    with open(tsv_path, "w", encoding="utf-8") as tsv_out, \
         open(fasta_path, "w", encoding="utf-8") as fasta_out:
        print(*columns, sep="\t", file=tsv_out)

        # Retrieve and parse one batch at a time.
        for batch, total in get_batch(search_url):
            batch_json = json.loads(batch.text)
            for entry in batch_json["results"]:
                n_total += 1
                if filter_function(entry):
                    n_filtered += 1

                    # Extract the fields and write one TSV row.
                    fields = extract_function(entry)
                    print(*fields, sep="\t", file=tsv_out)

                    # Write the same protein to the FASTA file.
                    accession = entry["primaryAccession"]
                    sequence = entry["sequence"]["value"]
                    print(f">{accession}", sequence, sep="\n", file=fasta_out)

    print(f"Done! Total retrieved: {n_total}, passed filter: {n_filtered}")
    print(f"TSV saved to: {tsv_path}")
    print(f"FASTA saved to: {fasta_path}")

## 3. Collect the preliminary datasets

We call `get_dataset` once for positive data and once for negative data, using the appropriate filters and extraction functions. This produces the four files requested on slides 29-32.

| Dataset | Metadata | Protein sequences |
| --- | --- | --- |
| Positive | `positive_dataset.tsv` | `positive_dataset.fasta` |
| Negative | `negative_dataset.tsv` | `negative_dataset.fasta` |

The positive TSV includes the cleavage position; the negative TSV includes the TM flag. Both include accession, organism, kingdom, and protein length.

After execution, check the retrieved and retained counts and confirm that each TSV and its FASTA contain the same proteins. These are the preliminary data for the subsequent preparation and analysis steps.

> Files are saved in `data/collected/` relative to the repository root. The code locates the root from the current working directory, so it works from the root or from `notebooks/`. Running this cell again overwrites the collected datasets, and counts may change when UniProt is updated.

In [ ]:
# Find the repository root from the current working directory.
project_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "notebooks" / "02a_data_collection.ipynb").is_file():
        project_root = folder
        break
if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the LB2 project repository.")

output_dir = project_root / "data" / "collected"
output_dir.mkdir(parents=True, exist_ok=True)

positive_columns = ["accession", "organism", "kingdom", "length", "cleavage_pos"]
negative_columns = ["accession", "organism", "kingdom", "length", "has_tm_in_90"]

# Positive dataset: proteins with a valid experimental signal peptide.
get_dataset(
    positive_url, filter_positive_entry, extract_positive_fields,
    positive_columns, output_dir / "positive_dataset"
)

# Negative dataset: proteins without any signal peptide annotation.
get_dataset(
    negative_url, filter_negative_entry, extract_negative_fields,
    negative_columns, output_dir / "negative_dataset"
)